# Wikimedia Bronze Layer

Generates raw Wikimedia event data via `produce()` (defined in `Producer_streaming_engine.ipynb`), converts it into a Spark DataFrame, and persists it to the bronze Delta table `workspace.bronze.wikimedia_bronze`. The `read_raw()` helper adds an `_ingest_time` timestamp, casts the `timestamp` column, caches the DataFrame, and appends to `{catalog}.{bronze_schema}.{bronze_table}` using a batch write. Configurations — catalog, schema, and table names — are sourced from `params_config` via `%run`.

In [0]:
%run ./params_config

In [0]:
%run ./Producer_streaming_engine.ipynb

In [0]:
def read_raw(spark_df):
    # Add ingest timestamp, cast timestamp, persist to bronze, and hand off the same rows downstream
    df = (spark_df
          .withColumn("_ingest_time", current_timestamp())
          .withColumn("timestamp", col("timestamp").cast("timestamp"))
          .cache())

    (df.write
       .format("delta")
       .mode("append")
       .saveAsTable(f"{catalog}.{bronze_schema}.{bronze_table}"))

    return df


In [0]:
# Convert the pandas DataFrame from produce() into a Spark DataFrame
pdf = produce(max_events=100)
spark_df = spark.createDataFrame(pdf)
print(f"Spark DataFrame row count: {spark_df.count()}")
df=read_raw(spark_df)
display(df)